# Day 12(M3 Day03) 실습 — pgvector·RAG로 내 문서 기반 답변 만들기

**목표**: 문서를 청킹해 pgvector에 저장·검색하고, 리트리버(Retriever)+프롬프트+LLM을 체인으로 묶어 근거 기반으로 답한다.
**구성**: Part 1 청킹·저장·검색(+청킹/연결 오류 2종) → Part 2 청크 크기 실험(+사내 복리후생 문서 도메인 반복) → Part 3 RAG 체인(+Day01~02 면접 코치를 pgvector로 업그레이드) → Part 4 PDF 문서로 RAG(불러오기·잡음 정리·쪽 번호 근거·메타데이터 필터)

> **참고:** 실습 전 가상환경 활성화, `.env`의 `OPENAI_API_KEY`·`DATABASE_URL`을 확인한다. `uv add langchain-postgres langchain-text-splitters pymupdf`와 pgvector 확장(`CREATE EXTENSION vector`)이 필요하다(이미 준비돼 있다면 생략). Part 4는 `data/` 폴더에 PDF 3종이 있어야 한다(배포 문서 「실습용 PDF 내려받기」).

## 0. 환경 준비 (연결·모델)

In [ ]:
import os
os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGCHAIN_TRACING_V2"] = "false"

from langchain_postgres import PGVector
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv

load_dotenv()
PGVECTOR_URL = os.getenv("DATABASE_URL").replace("postgresql://", "postgresql+psycopg://")

print("준비 완료")

In [7]:
from sqlalchemy import create_engine
pg_engine = create_engine(PGVECTOR_URL, pool_size=2, max_overflow=0)

In [8]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4o-mini")

## Part 1. 청킹·저장·검색 — 기초

완성 코드를 직접 쳐서 문서를 자르고 pgvector에 저장한 뒤, 리트리버로 관련 조각을 찾아본다.

> **참고:** M1 Day01에서 프롬프트에 넣었던 역할·지시문·맥락(`context`)·예시는 전부 **정적 콘텐츠**였다 — 미리 정해서 프롬프트에 고정으로 박아둔 것이다. 오늘부터는 그 `context`를 하드코딩하는 대신, 질문마다 **그때그때 검색해서 찾아온 문서**로 채운다 — 이게 **동적 콘텐츠**이고, RAG(Retrieval-Augmented Generation)의 핵심이다.

### 1-1. 문서 준비·청킹

긴 문서를 검색하기 좋은 크기로 자른다.

In [10]:
DOC = """LangChain은 LLM 애플리케이션을 만드는 프레임워크다.
LCEL은 파이프 기호로 부품을 연결하는 방식이다.
임베딩은 문장을 숫자 벡터로 바꾼다.
pgvector는 PostgreSQL에서 벡터를 저장하고 검색하는 확장이다.
RAG는 관련 문서를 찾아 근거로 답하는 방식이다."""

# TODO: chunk_size=50, chunk_overlap=0인 RecursiveCharacterTextSplitter로 DOC을 잘라 chunks에 저장하세요
splitter = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=0)
chunks = splitter.split_text(DOC)
len(chunks)

4

In [11]:
type(chunks)

list

In [14]:
for c in chunks:
    print(c)
    print('-' * 50)

LangChain은 LLM 애플리케이션을 만드는 프레임워크다.
--------------------------------------------------
LCEL은 파이프 기호로 부품을 연결하는 방식이다.
임베딩은 문장을 숫자 벡터로 바꾼다.
--------------------------------------------------
pgvector는 PostgreSQL에서 벡터를 저장하고 검색하는 확장이다.
--------------------------------------------------
RAG는 관련 문서를 찾아 근거로 답하는 방식이다.
--------------------------------------------------


### 1-2. pgvector에 저장

PGVector가 임베딩·저장을 한 번에 처리한다.

In [15]:
# TODO: embeddings·collection_name("my_docs")·connection(engine)으로 PGVector store를 만들고
#       (pre_delete_collection=True, use_jsonb=True) chunks를 add_texts로 저장하세요

store = PGVector(
    embeddings=embeddings, # embedding 모델 객체
    collection_name="my_docs", #저장소 이름
    connection=pg_engine, # pgvector connection
    use_jsonb=True,
    pre_delete_collection=True #실습중 켜기
)

store.add_texts(chunks)

print("저장 완료")

Collection not found


저장 완료


### 1-3. 유사도 검색

질문과 가까운 조각을 찾는다.

In [18]:
# TODO: store.similarity_search로 "pgvector가 뭐야?"와 가까운 조각 2개를 hits에 찾으세요

query = "pgvector가 뭐야?"

hits = store.similarity_search(query, k=1)
for h in hits:
    print("결과: ", h.page_content)

결과:  pgvector는 PostgreSQL에서 벡터를 저장하고 검색하는 확장이다.


In [ ]:
hits = store.similarity_search_with_score(query, k=4)
for h, score in hits:
    print("결과: ", h.page_content, score) # score는 거리값, 가까우면 순서가 높다.

결과:  pgvector는 PostgreSQL에서 벡터를 저장하고 검색하는 확장이다. 0.340888127060019
결과:  LCEL은 파이프 기호로 부품을 연결하는 방식이다.
임베딩은 문장을 숫자 벡터로 바꾼다. 0.775669009972781
결과:  RAG는 관련 문서를 찾아 근거로 답하는 방식이다. 0.807979745211868
결과:  LangChain은 LLM 애플리케이션을 만드는 프레임워크다. 0.811061178305428


### 1-4. 리트리버(retriever)로 변환

체인에 꽂을 수 있는 리트리버로 바꾼다.
기본체인: prompt | llm | parser -> Runnable

In [ ]:
# TODO: store.as_retriever(search_keywords={"k": 2})로 retriever를 만들고
#       "임베딩이 뭐야?"를 invoke해 docs에 저장하세요
# hits = store.similarity_search(query, k=1)

retriever = store.as_retriever(search_keywords={"k": 2})
ret_result = retriever.invoke("임베딩이 뭐야?")
ret_result

[Document(id='5e2e4760-7a5b-43dc-a159-2a1ca35c1e5b', metadata={}, page_content='LCEL은 파이프 기호로 부품을 연결하는 방식이다.\n임베딩은 문장을 숫자 벡터로 바꾼다.'),
 Document(id='57254241-b1fe-4759-b181-78676b3b502f', metadata={}, page_content='LangChain은 LLM 애플리케이션을 만드는 프레임워크다.'),
 Document(id='29b9ec5d-9e33-4d64-be49-df93bd9009b5', metadata={}, page_content='pgvector는 PostgreSQL에서 벡터를 저장하고 검색하는 확장이다.'),
 Document(id='7b09f857-969c-4fad-a2f0-d4cb5cc65af7', metadata={}, page_content='RAG는 관련 문서를 찾아 근거로 답하는 방식이다.')]

### 1-5. 오류 대응 ① — chunk_overlap이 chunk_size보다 크면?

겹침(overlap)을 자르는 크기보다 크게 주면 어떻게 되는지 확인한다.

In [25]:
# TODO: chunk_size=50, chunk_overlap=60인 RecursiveCharacterTextSplitter를 만들어 DOC을 잘라보세요
splitter = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=10, separators=[""])
chunks = splitter.split_text(DOC)

for c in chunks:
    print(c)
    print('-'*50)

LangChain은 LLM 애플리케이션을 만드는 프레임워크다.
LCEL은 파이프 기호로 부
--------------------------------------------------
파이프 기호로 부품을 연결하는 방식이다.
임베딩은 문장을 숫자 벡터로 바꾼다.
pgvec
--------------------------------------------------
바꾼다.
pgvector는 PostgreSQL에서 벡터를 저장하고 검색하는 확장이다.
RA
--------------------------------------------------
는 확장이다.
RAG는 관련 문서를 찾아 근거로 답하는 방식이다.
--------------------------------------------------


> **주의:** 결과는 실제 실행에서 직접 확인한다. `chunk_overlap`은 반드시 `chunk_size`보다 작아야 한다 — 겹치는 부분이 조각 자체보다 클 수는 없기 때문이다.

### 1-6. 실측 확인 — 연결 스킴을 안 바꾸면?

`DATABASE_URL`을 `postgresql+psycopg://`로 바꾸지 않고 그대로 PGVector에 넘기면 어떻게 되는지 직접 실행해서 확인한다.

In [ ]:

    # TODO: connection에 os.getenv("DATABASE_URL")을 스킴 변환 없이 그대로 넣어 PGVector를 만들고
    #       ["테스트 문장"]을 add_texts로 저장해보세요




> **참고:** 실제로 실행하면(이 환경·버전 기준) 스킴을 안 바꿔도 **오류 없이 저장·검색까지 정상 동작한다** — `langchain-postgres`가 내부적으로 스킴을 알아서 처리하기 때문으로 보인다. 문서(1-1 셋업 셀)가 안내하는 `postgresql+psycopg://` 변환이 '반드시 필요한' 것은 아닐 수 있다는 뜻이다. 다만 이는 라이브러리 버전에 따라 달라질 수 있는 동작이므로, 공식적으로 안내된 스킴 변환은 그대로 지키는 것이 안전하다 — **직접 실행해서 확인하기 전까지는 문서의 '반드시 이렇게 해야 한다'는 설명도 가정일 뿐**이라는 점이 이번 확인의 핵심이다.

### 관찰 정리

- 저장한 조각을 질문으로 어떻게 찾아 오나?
- 1-5의 오류는 어떤 값을 잘못 넣었을 때 생겼나?
- 1-6에서 확인한 것처럼, 문서의 '반드시 이렇게 해야 한다'는 설명도 실제로 돌려 보기 전까지는 가정일 뿐이라는 원칙은 왜 중요한가?

## Part 2. 청크 크기와 검색 품질 — 응용

자르는 크기(chunk_size)에 따라 검색 결과가 어떻게 달라지는지 관찰한다.

### 2-1. 긴 문서와 저장 함수

크기·컬렉션 이름을 받아 저장하는 함수를 만든다.

In [26]:
LONG = ("파이썬은 1991년 귀도 반 로섬이 발표한 프로그래밍 언어다. "
        "문법이 간결해 배우기 쉽다. "
        "데이터 분석과 인공지능 분야에서 널리 쓰인다. "
        "라이브러리 생태계가 방대하다. "
        "인터프리터 언어라 실행 전 컴파일이 필요 없다.")

# TODO: size(chunk_size)·name(collection_name)을 받아 LONG을 잘라 PGVector에 저장하고
#       (store, chunks)를 반환하는 build 함수를 만드세요

def build(size, name):
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=0)
    chunks = splitter.split_text(LONG)

    s = PGVector(
        embeddings=embeddings,
        collection_name=name,
        connection=pg_engine,
        use_jsonb=True,
        pre_delete_collection=True
    )
    s.add_texts(chunks)
    return s, chunks

In [39]:
s_test, sc = build(30, "small_test")


In [40]:
l_test, lc = build(100, "long_test")

In [41]:
q = "파이썬은 언제 만들어졌나?"
s_result = s_test.similarity_search(q, k=1)
l_result = l_test.similarity_search(q, k=1)

In [42]:
for h in s_result:
    print(h.page_content)

파이썬은 1991년 귀도 반 로섬이 발표한 프로그래밍


In [43]:
for h in l_result:
    print(h.page_content)

파이썬은 1991년 귀도 반 로섬이 발표한 프로그래밍 언어다. 문법이 간결해 배우기 쉽다. 데이터 분석과 인공지능 분야에서 널리 쓰인다. 라이브러리 생태계가 방대하다. 인터프리터


### 2-2. 작은 청크(30) vs 큰 청크(120)

같은 문서를 서로 다른 크기로 저장한다.

In [ ]:
# TODO: build(30, "small_chunks")와 build(120, "large_chunks")로 small, sc / large, lc를 만드세요



### 2-3. 같은 질문으로 검색 비교

In [ ]:
q = "파이썬은 언제 만들어졌나?"


### 관찰

| 크기 | 조각 수 | 장점 | 단점 |
| --- | --- | --- | --- |
| 작게(30) | 여러 개 | 정확히 겨냥 | 앞뒤 문맥 부족 |
| 크게(120) | 적음 | 문맥 풍부 | 군더더기 섞임 |

### 2-4. 다른 도메인 반복 — 사내 복리후생 안내문

같은 방법을 다른 도메인에도 적용해본다. 이번엔 사내 복리후생 안내문으로 청크 크기 비교를 반복한다.

In [ ]:
BENEFIT = ("우리 회사는 전 직원에게 연 100만원의 자기계발비를 지원한다. "
           "도서 구입, 온라인 강의, 자격증 시험 응시료에 사용할 수 있다. "
           "매년 1월에 사용 한도가 초기화되며 이월되지 않는다. "
           "신청은 사내 포털에서 영수증을 첨부해 진행한다. "
           "승인까지 보통 3영업일이 걸린다.")



# TODO: build_benefit(30, "benefit_small")와 build_benefit(120, "benefit_large")로
#       b_small, bsc / b_large, blc를 만드세요



### 관찰 정리

- 파이썬 도메인과 복리후생 도메인 모두에서, 작은 청크·큰 청크의 장단점이 그대로 나타났는가?
- 실무 문서라면 어떤 크기가 더 적당할지 어떻게 정할 수 있을까?

## Part 3. 미니 프로젝트 — RAG 체인

리트리버 + 프롬프트 + LLM을 체인으로 묶어 문서 기반으로 답한다.

### 3-1. 문서 인덱싱

자르기 → 저장 → 리트리버 변환을 한 번에 해 둔다.

In [ ]:
DOC = """LangChain은 LLM 애플리케이션을 만드는 프레임워크다.
LCEL은 파이프 기호로 부품을 연결하는 방식이다.
임베딩은 문장을 숫자 벡터로 바꾼다.
pgvector는 PostgreSQL에서 벡터를 저장하고 검색하는 확장이다.
RAG는 관련 문서를 찾아 근거로 답하는 방식이다."""

# TODO: DOC을 청킹해 chunks를 만들고, PGVector store(collection_name="rag_docs")에 저장한 뒤
#       search_kwargs={"k": 2}인 retriever로 변환하세요




print("인덱싱 완료")

### 3-2. RAG 체인 구성

검색된 조각을 `{context}`로, 질문을 `{question}`으로 넣는다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# TODO: '근거에 없으면 모른다고 답해라'는 지시를 담은 prompt를 ChatPromptTemplate.from_template로 만드세요
#       ({context}, {question} 변수 사용)



# TODO: retriever | format_docs를 context로, RunnablePassthrough()를 question으로 하는
#       chain을 prompt | llm | StrOutputParser()까지 연결해 만드세요


print("체인 준비 완료")

### 3-3. 질문·환각 확인

문서에 있는 질문은 답하고, 없는 질문은 모른다고 해야 정상이다.

In [ ]:

# TODO: 문서에 없는 질문("파이썬 창시자가 누구야?")을 chain에 invoke해보세요

### 3-4. 근거 확인

답변이 어떤 조각을 근거로 했는지 함께 확인한다.

In [ ]:
# 근거 확인: 어떤 조각을 참고했나


### 관찰

| 질문 | 문서에 있나 | 결과 |
| --- | --- | --- |
| pgvector가 뭐야? | 있음 | 근거로 답함 |
| RAG가 뭐야? | 있음 | 근거로 답함 + 출처 확인 |
| 파이썬 창시자? | 없음 | 모른다 |

## Part 3-확장. Day01~02 통합 — 면접 코치를 pgvector로 업그레이드

Day01(M3)에서는 채용 공고를 파이썬 리스트+수동 코사인 유사도로 검색했다. 오늘은 같은 채용 공고를 **pgvector에 저장**하고 `retriever`로 검색하도록 바꾼다. Day02(M3)의 세션 저장(`save`/`load`)과 M1 Day04의 방어 로직은 그대로 재사용한다.

### 세션 저장소 연결 (Day02 재사용)

In [ ]:
# TODO: Day02(M3)의 psycopg2 연결과 save·load 함수를 그대로 옮겨오세요
#       (session_messages 테이블은 Day02에서 이미 만들어져 있다.
#        1과목 챗봇의 messages 테이블과 다른 테이블이므로 이름을 바꾸지 않는다)

### 채용 공고를 pgvector에 저장 (Day01 업그레이드)

In [ ]:
job_postings = [
    "백엔드 개발자: Python/Java 기반 서버 개발, RESTful API 설계, DB 최적화 경험 우대",
    "프론트엔드 개발자: React/TypeScript, 반응형 UI 구현, 웹 접근성 이해 우대",
    "데이터 분석가: SQL/Python 데이터 분석, 대시보드 제작, 통계적 사고 우대",
    "마케터: 콘텐츠 기획, 데이터 기반 캠페인 분석, SNS 채널 운영 경험 우대",
]
# TODO: job_postings를 collection_name="job_postings"인 PGVector store에 저장하고
#       (pre_delete_collection=True) search_kwargs={"k": 1}인 job_retriever로 변환하세요






print("채용 공고 pgvector 저장 완료")

### 방어 로직 (M1 Day04 재사용)

인젝션 방어는 M1 Day04에서 만든 것을 그대로 가져온다.

In [ ]:
# TODO: M1 Day04의 위험 문구·금지어 목록과 input_guard·output_guard를 옮겨오세요






### pgvector 기반 세션 면접 코치

Day02(M3)의 `session_coach`와 구조는 같지만, 채용 공고 검색을 수동 코사인 유사도 대신 `job_retriever`(pgvector)로 바꾼다. Day02에서 배운 대로, 검색 결과는 참고 자료일 뿐이라는 지시는 그대로 유지한다.

In [ ]:

    # TODO: job_retriever로 message와 관련된 채용 공고를 related_docs에서 찾아 related_job을 만들고,
    #       (참고 자료일 뿐이라는 지시를 포함한) SystemMessage + 대화 기록으로 msgs를 만든 뒤
    #       llm.invoke(msgs).content로 answer를 만드세요




### 최종 테스트

In [ ]:
# 다시 실행할 때 이전 실행의 대화가 섞이지 않도록 테스트 세션을 비운다

# TODO: 인젝션을 시도하는 질문을 넣어보세요

## Part 4. PDF 문서로 RAG — 쪽 번호까지 근거 제시

지금까지는 코드에 적은 짧은 문장을 문서로 썼다. 실제 보고서 PDF 3종으로 같은 과정을 밟는다. 새로 생기는 일은 세 가지다.

| 새로 생기는 일 | 절 |
| --- | --- |
| 파일에서 텍스트를 꺼낸다 | 4-1 |
| 쪽마다 반복되는 머리말·쪽 번호를 지운다 | 4-2 |
| 어느 파일 몇 쪽인지(메타데이터)를 답변까지 들고 간다 | 4-3 ~ 4-6 |

> **참고:** PDF는 배포 문서 「실습용 PDF 내려받기」대로 `data/` 폴더에 받아 둔다. NIPA 이용 규칙상 제3자 배포가 금지되어 저장소에는 PDF가 없다.

### 4-1. PDF 불러오기 — 쪽마다 Document 하나

`pymupdf`로 PDF를 열어 쪽마다 텍스트를 꺼내고, 파일명과 쪽 번호를 `metadata`에 담는다. `Document`는 LangChain에서 문서 조각을 담는 그릇이다(`page_content` + `metadata`).

In [ ]:

        # TODO: 쪽(page)마다 Document를 만들어 리스트로 반환하세요
        #       page_content=page.get_text(), metadata={"source": 파일명, "page": 1부터 센 쪽 번호}



> **참고:** LangChain에도 같은 일을 하는 `PyMuPDFLoader`가 있다. 다만 이 로더가 들어 있는 `langchain-community` 패키지는 지원 종료 예정이라 불러올 때 경고가 나온다. 로더가 하는 일은 위 함수가 전부이므로 직접 만들어 쓴다. `PyMuPDFLoader`의 `page`는 0부터 센다는 점도 다르다.

### 4-2. 쪽마다 반복되는 잡음 정리

4-1 출력의 앞부분을 보면 본문 앞에 `국가별 ICT 시장동향`(머리말)과 `7`(쪽 번호)이 붙어 있다. 모든 쪽에 반복되므로 검색에 방해만 된다. 또 양쪽 정렬 때문에 문장 중간에 줄바꿈(`\n`)이 들어가 있다.

In [ ]:

    # TODO: HEADER에 맞는 부분을 지우고, 줄바꿈(앞뒤 공백 포함)을 공백 하나로 바꾸세요



> **주의:** "숫자만 있는 줄을 모두 지운다"로 쪽 번호를 없애면 표 안의 숫자 칸(연도, 금액)까지 지워진다. 머리말 바로 아래 줄만 지우도록 범위를 좁힌다. 정리 규칙은 문서마다 다르므로, 먼저 실제 텍스트를 출력해 보고 정한다.

### 4-3. 청킹 — split_documents는 메타데이터를 지킨다

Part 1은 `split_text`로 문자열을 잘랐다. 문서가 `Document`이면 `split_documents`를 쓴다. 잘린 조각마다 원래 쪽의 `metadata`가 그대로 복사된다.

이번에는 `from_tiktoken_encoder`로 **토큰 수** 기준으로 자른다. Part 1의 `chunk_size`는 글자 수였다.

In [ ]:
# TODO: all_pages를 메타데이터가 유지되도록 잘라 pdf_chunks에 담으세요


### 4-4. pgvector에 저장

Part 1과 같은 `PGVector`에 `add_documents`로 넣는다. `metadata`도 함께 저장된다(`use_jsonb=True`).

> **참고:** 처음 실행하면 `Collection not found`가 한 번 출력된다. `pre_delete_collection=True`가 지울 컬렉션을 찾았는데 아직 없다는 뜻이고, 오류가 아니다.

### 4-5. 근거 쪽 번호와 함께 답하기

3-4에서는 답변과 근거를 따로 출력했다. 이번에는 근거마다 `[파일명 쪽]`을 붙여 프롬프트에 넣고, 답변 문장에도 그 표시를 붙이게 한다.

In [ ]:

    # TODO: 조각마다 "[파일명 N쪽]" 한 줄과 본문을 붙이고, 조각 사이는 빈 줄로 이어 반환하세요

pdf_prompt = ChatPromptTemplate.from_template(
    "다음 근거만 사용해 답한다.\n"
    "문장마다 끝에 근거의 [파일명 쪽] 표시를 그대로 붙인다.\n"
    "근거에 없는 내용이면 '문서에 없습니다'라고만 답하고 표시를 붙이지 않는다.\n\n"
    "근거:\n{context}\n\n질문: {question}"
)


In [ ]:
# 답변의 [파일명 쪽] 표시가 실제 검색 결과와 맞는지 확인


### 4-6. 오류 대응 — 검색 결과를 그대로 프롬프트에 넣으면?

`{"context": pdf_retriever}`처럼 `format_docs` 없이 리트리버를 바로 연결하면, 검색 결과(`Document` 리스트)가 문자열로 바뀌어 그대로 들어간다. 프롬프트에 실제로 무엇이 들어가는지 비교한다.

In [ ]:
q = "글로벌 블록체인 시장 규모 전망은?"


오류는 나지 않는다. 대신 `Document(id=…, metadata={…}, page_content='…')` 같은 파이썬 표현이 섞여 들어가 토큰을 낭비하고, 쪽 번호가 문장 속에 묻혀 모델이 출처 표시를 제대로 붙이지 못한다. 리트리버 뒤에는 항상 문자열로 바꾸는 함수를 둔다.

### 4-7. 메타데이터 필터 — 파일 하나에서만 찾기

`filter`로 `metadata` 조건을 걸면 그 조건에 맞는 조각 안에서만 검색한다. 같은 질문을 필터 없이, 그리고 일본 보고서로만 좁혀 검색한다.

In [ ]:
q = "정부의 AI 정책"


### 관찰

| 확인 | 결과 |
| --- | --- |
| 4-2 잡음 정리 | 머리말·쪽 번호가 사라지고 문장이 한 줄로 이어진다 |
| 4-3 `split_documents` | 조각마다 `source`·`page`가 남는다. `split_text`는 문자열만 남긴다 |
| 4-5 답변 | 문장 끝에 `[파일명 쪽]`이 붙는다. 문서에 없는 질문은 "문서에 없습니다"라고만 답한다 |
| 4-6 그대로 넣기 | 오류 없이 `Document(`·`metadata=`가 프롬프트에 섞인다 |
| 4-7 필터 | 필터가 없으면 미국 보고서 위주로 나오고, 필터를 걸면 일본 보고서에서만 나온다 |

> **참고:** Day14 보충 실습(`M4-흐름제어/Day14-보충-에이전틱RAG-실습.ipynb`)에서는 4-7의 필터를 파일별 **검색 도구**로 만들어, 질문에 맞는 보고서를 에이전트가 고르게 한다.

### Day 12(M3 Day03) 정리

**확인 질문**
- 수동 코사인 유사도(Day01)와 pgvector 리트리버(오늘)는 코치 입장에서 무엇이 달라졌고 무엇이 그대로인가?
- 이 RAG(장기 기억)에 Day02의 세션 메모리(단기 기억)를 더한 지금 구조가, M4에서 배울 Hybrid Memory와 어떻게 이어질지 예상해보자.
- PDF에서 꺼낸 조각에 파일명·쪽 번호를 남겨 두면, 답변의 근거를 사람이 원문에서 직접 확인할 수 있다. 이것이 왜 중요한가?

## 확인 문제

1. RAG 6단계 흐름을 순서대로 말하면?
2. 1-5의 오류는 무엇을 잘못 설정했을 때 생겼나? 1-6은 왜 '오류'가 아니라 '실측 확인'이었나?
3. Part 2와 Part 2의 도메인 반복에서, 작은 청크·큰 청크의 장단점이 도메인이 달라도 그대로 통했는가?
4. "근거에 없으면 모른다"는 지시가 없다면 3-3의 결과는 어떻게 달라졌을까?
5. `rag_session_coach`에서 Day01·Day02·M1 Day04·오늘 각각에서 가져온 요소는 무엇인가?
6. 4-2에서 "숫자만 있는 줄을 모두 지우기" 대신 머리말 바로 아래 줄만 지운 이유는?
7. `split_text`와 `split_documents`는 결과에서 무엇이 다른가?
8. 4-6처럼 검색 결과를 그대로 프롬프트에 넣으면 오류가 나지 않는데도 왜 문제인가?